In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ooodg481/unsloth_d1eb7abc121e4d798ae8c07500451007
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ooodg481/unsloth_d1eb7abc121e4d798ae8c07500451007
  Resolved https://github.com/unslothai/unsloth.git to commit 9a3ee8777cd71cfb07fad1b4319d8c35cbbca9cd
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto-detect (Float16 for T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.14: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth supports 0 for maximum speed
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves 30% VRAM
    random_state = 3407,
)

Unsloth 2026.9.14 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [11]:
import json
from datasets import Dataset

system_prompt = (
    "You are BharatFinanceEdu, an expert Indian financial educator. "
    "Your goal is to explain financial concepts simply, using clear analogies, "
    "practical examples, and dispelling common misconceptions. "
    "Always respond in the language asked."
)

formatted_data = []
skipped_lines = 0

with open("final_train.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            skipped_lines += 1
            continue # Safely skips the smashed line causing your error

        # Build structured response body
        answer_parts = [f"**Explanation:** {item.get('simple_answer', '')}"]
        if item.get("analogy"):
            answer_parts.append(f"**Analogy:** {item['analogy']}")
        if item.get("example"):
            answer_parts.append(f"**Example:** {item['example']}")
        if item.get("misconception"):
            answer_parts.append(f"**Common Misconception:** {item['misconception']}")

        full_assistant_reply = "\n\n".join(answer_parts)

        # Apply standard conversational format
        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": item.get("question", "")},
            {"role": "assistant", "content": full_assistant_reply}
        ]

        try:
            formatted_text = tokenizer.apply_chat_template(
                conversation,
                tokenize=False,
                add_generation_prompt=False
            )
            formatted_data.append({"text": formatted_text})
        except Exception:
            skipped_lines += 1
            continue

dataset = Dataset.from_list(formatted_data)
print(f"Successfully loaded and formatted {len(dataset)} training examples.")
print(f"Skipped {skipped_lines} corrupted lines.")

Successfully loaded and formatted 4085 training examples.
Skipped 1 corrupted lines.


In [12]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150, # Fast 15-minute run for the hackathon
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# This line actually starts the GPU training
trainer_stats = trainer.train()

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4085 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,085 | Num Epochs = 1 | Total steps = 150
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:5429: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.342705
20,1.560736
30,1.212934
40,1.170818
50,1.139660
60,1.116367
70,1.045451
80,1.149841
90,1.138530
100,1.190046


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.


In [13]:
model.save_pretrained_merged("bharat_finance_model", tokenizer, save_method="lora")
!zip -r bharat_finance_model.zip bharat_finance_model

Unsloth: Restored added_tokens_decoder metadata in bharat_finance_model/tokenizer_config.json.


Unsloth: Saving tokenizer... Done.
Unsloth: Saving model... Done.
  adding: bharat_finance_model/ (stored 0%)
  adding: bharat_finance_model/README.md (deflated 65%)
  adding: bharat_finance_model/tokenizer_config.json (deflated 89%)
  adding: bharat_finance_model/tokenizer.json (deflated 81%)
  adding: bharat_finance_model/chat_template.jinja (deflated 71%)
  adding: bharat_finance_model/adapter_config.json (deflated 60%)
  adding: bharat_finance_model/adapter_model.safetensors (deflated 7%)


In [16]:
# Enable native 2x faster inference
FastLanguageModel.for_inference(model)

def test_model(question):
    # We use the exact same system prompt from the training step
    messages = [
        {"role": "system", "content": (
            "You are BharatFinanceEdu, an expert Indian financial educator. "
            "Your goal is to explain financial concepts simply, using clear analogies, "
            "practical examples, and dispelling common misconceptions. "
            "Always respond in the language asked."
        )},
        {"role": "user", "content": question}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.3, # Low temperature keeps financial answers grounded
        use_cache=True
    )

    # Decode only the newly generated tokens
    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

    print(f"Question: {question}")
    print("-" * 40)
    print(response)
    print("=" * 60 + "\n")

# Test 1: Core financial concept (English)
test_model("What is compound interest and why is it important?")

# Test 2: Common investment vehicle (Hindi)
test_model("चक्रवृद्धि ब्याज (Compound Interest) क्या है और यह क्यों महत्वपूर्ण है?")

# Test 3: Market basics (Marathi)
test_model("चक्रवाढ व्याज (Compound Interest) म्हणजे काय आणि ते का महत्त्वाचे आहे?")

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is compound interest and why is it important?
----------------------------------------
**Explanation:** Compound interest means earning interest on your interest, so your money grows faster over time.

**Analogy:** It's like a snowball rolling down a hill: the bigger it gets, the faster it rolls.

**Example:** If you save ₹100 each month at 12% interest, after 30 years you have ₹85,000 instead of ₹36,000 without compound interest.

**Common Misconception:** Many think interest only applies to the principal amount, but compound interest also earns interest on the interest earned earlier.



Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: चक्रवृद्धि ब्याज (Compound Interest) क्या है और यह क्यों महत्वपूर्ण है?
----------------------------------------
**Explanation:** चक्रवृद्धि ब्याज एक राशि पर अन्य राशि पर ब्याज लगाने से उत्पन्न होता है। इसका मतलब है कि ब्याज भुगतान के बाद प्रत्येक वर्ष या विमान में बढ़ता है।

**Analogy:** यह एक गौरवपूर्ण फूल की तरह है जो बढ़ता है और अपने आप में बढ़ता है, जिससे फूल बढ़ता है और उसके आगे फूल बढ़ता है।

**Example:** यदि 1000 रुपये पर 5% चक्रवृद्धि ब्याज की दर से ब्याज लगाया जाता है, तो ब्याज राशि प्रत्येक वर्ष बढ़ती है: पहले वर्ष 50 रुपये, दूसरे वर्ष 52.50 रुपये, और इस प्रकार जारी रहती है।

**Common Misconception:** लोग अक्सर सोचते हैं कि ब्याज एक बार बढ़ा ही रहता है, लेकिन चक्रवृद्धि ब्याज में ब्याज राशि प्रत्येक वर्ष या विमान में बढ़ती है।

Question: चक्रवाढ व्याज (Compound Interest) म्हणजे काय आणि ते का महत्त्वाचे आहे?
----------------------------------------
**Explanation:** चक्रवाढ व्याज हे एखाद्या समयानंतर वाढलेल्या बँडांची वृद्धी असलेली राशी आहे, ज्यामुळे बँडांची वृद्धी वा

In [17]:
def generate_quiz(topic, num_questions=5):
    # A new, strict system prompt forcing a quiz format
    quiz_system_prompt = (
        "You are a strict financial quiz master. Your task is to generate a multiple-choice "
        "quiz on the requested topic. Do NOT use the Explanation/Analogy format. "
        "Output exactly in this format:\n"
        "1. [Question]\n"
        "A) [Option]\nB) [Option]\nC) [Option]\nD) [Option]\n"
        "Answer: [Correct Option]"
    )

    messages = [
        {"role": "system", "content": quiz_system_prompt},
        {"role": "user", "content": f"Generate a {num_questions}-question beginner quiz on {topic}."}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    # Slightly higher temperature (0.5) to allow for creative question generation
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=600,
        temperature=0.5,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]

    print(f"--- {topic.upper()} QUIZ ---")
    print(response)
    print("=" * 50)

# Test it out!
generate_quiz("Mutual Funds and Diversification", num_questions=5)
generate_quiz("शेअर बाजार (Stock Market in Marathi)", num_questions=3)

Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=600) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- MUTUAL FUNDS AND DIVERSIFICATION QUIZ ---
1. What is diversification?
A) Putting all your money into one stock
B) Investing only in government bonds
C) Spreading investments across different asset classes
D) Buying only mutual funds

2. Why do mutual funds diversify?
A) To guarantee profits
B) To reduce risk by spreading investments
C) To focus on one industry
D) To avoid taxes

3. What does diversification help with?
A) Guaranteeing high returns
B) Reducing risk through spread of investments
C) Eliminating market volatility
D) Avoiding any investment losses

4. How many stocks must you own to be diversified?
A) One
B) Two or more
C) At least 100
D) As many as possible

5. What is the main benefit of mutual fund diversification?
A) Guaranteed high returns
B) Reduced risk through spread of investments
C) Elimination of market volatility
D) Avoidance of investment losses
--- शेअर बाजार (STOCK MARKET IN MARATHI) QUIZ ---
**Explanation:** शेअर बाजारातील आपल्या पैस्याचा वापर करण्यासाठी 

In [19]:
# 1. Install yfinance inside Colab
!pip install -q yfinance

import yfinance as yf
from unsloth import FastLanguageModel

# Ensure the model is in fast inference mode
FastLanguageModel.for_inference(model)

# 2. Live stock helper function
def get_live_stock_data(ticker_symbol):
    try:
        if not ticker_symbol.endswith(".NS"):
            ticker_symbol += ".NS"
        stock = yf.Ticker(ticker_symbol)
        info = stock.info
        price = info.get("currentPrice", "N/A")
        beta = info.get("beta", "N/A")
        high_52 = info.get("fiftyTwoWeekHigh", "N/A")
        low_52 = info.get("fiftyTwoWeekLow", "N/A")

        if beta != "N/A":
            risk_level = (
                "High Volatility / High Risk" if float(beta) > 1.2 else
                "Moderate Risk (Moves with index)" if float(beta) > 0.8 else
                "Low Volatility / Conservative Risk"
            )
        else:
            risk_level = "Risk profile unavailable"

        return (
            f"[LIVE MARKET DATA - {ticker_symbol}]: "
            f"Current Price: ₹{price} | 52-Week High: ₹{high_52} | 52-Week Low: ₹{low_52} | "
            f"Volatility Metric (Beta {beta}): {risk_level}."
        )
    except Exception as e:
        return ""

# 3. Test Function simulating RAG + Live Stock + Model
def run_evaluation(user_question, simulated_rag_context="", ticker=""):
    print(f"\nEvaluating Question: '{user_question}'")

    live_data = get_live_stock_data(ticker) if ticker else ""
    if live_data:
        print(f"-> Fetched API: {live_data}")

    system_instruction = (
        "You are BharatFinanceEdu, an expert Indian financial educator. "
        "Explain financial concepts simply, using clear analogies, "
        "practical examples, and dispelling common misconceptions. "
        "Use any provided document context and live market data. "
        "Always respond in the language asked."
    )

    full_context = f"Official SEBI/RBI Guidelines:\n{simulated_rag_context}\n\n{live_data}"

    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"{full_context}\n\nUser Question: {user_question}"}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=350,
        temperature=0.3,
        use_cache=True
    )

    response = tokenizer.batch_decode(outputs[:, inputs.shape[1]:], skip_special_tokens=True)[0]
    print("\n" + "="*50)
    print(response)
    print("="*50 + "\n")

# --- TEST 1: Stock Question with Live API Data ---
run_evaluation(
    user_question="Is Tesla a risky investment right now, what is the risk rate?",
    simulated_rag_context="SEBI advises retail investors to assess risk tolerance, diversify, and not invest based solely on short-term price momentum.",
    ticker="RELIANCE.NS"
)

# --- TEST 2: Hindi Question ---
run_evaluation(
    user_question="म्यूचुअल फंड में एसआईपी (SIP) शुरू करने का क्या फायदा है?",
    simulated_rag_context="AMFI guidelines explain that Systematic Investment Plans average out market volatility through Rupee Cost Averaging."
)

Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Evaluating Question: 'Is Tesla a risky investment right now, what is the risk rate?'
-> Fetched API: [LIVE MARKET DATA - RELIANCE.NS]: Current Price: ₹1167.7 | 52-Week High: ₹1611.8 | 52-Week Low: ₹1160.8 | Volatility Metric (Beta 0.152): Low Volatility / Conservative Risk.


Both `max_new_tokens` (=350) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



**Explanation:** The beta of Tesla is 0.93, which means it moves about 93% as much as the market. If the market goes up 10%, Tesla is expected to go up 9.3%. If the market goes down 10%, Tesla is expected to go down 9.3%. So Tesla is slightly more volatile than the market but not by much.


Evaluating Question: 'म्यूचुअल फंड में एसआईपी (SIP) शुरू करने का क्या फायदा है?'

**Explanation:** एसआईपी से आपको बाजार की विचलनता को नियमित रूप से और अधिक सुरक्षित ढंग से दूर किया जा सकता है।

**Analogy:** यह एक बाल्टी खरीदने की तरह है, जिसमें एक बार बाल्टी खरीदी जाती है और उसे एक साथ खाया जाता है, बजाय एक बार एक बड़ी राशि का खर्च करना।

**Example:** एक नव-प्रवेशी नियमित रूप से एसआईपी के माध्यम से एक छोटी राशि का निवेश करता है, जिससे बाजार की विचलनता को दूर किया जा सकता है।

**Common

